In [30]:
2**10

1024

In [31]:
import timm

model = timm.create_model(
    "resnet18",
    pretrained=True,
    num_classes=10
)

In [32]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act1): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (drop_block): Identity()
      (act1): ReLU(inplace=True)
      (aa): Identity()
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (act2): ReLU(inplace=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (b

In [33]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchinfo import summary
import torch
import torch.nn as nn
from tqdm import tqdm


transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
])

train_dataset = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)   

test_dataset = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset,
                          batch_size=64,
                          shuffle=True)

test_loader = DataLoader(test_dataset,
                         batch_size=64,
                         shuffle=False)

In [34]:
# torch.set_num_threads(8)
# print(torch.get_num_threads())
# print(torch.__config__.show())



In [35]:
# shapes of the data
for images, labels in train_loader:
    print(f"Images batch shape: {images.size()}")
    print(f"Labels batch shape: {labels.size()}")
    break

# total number of samples in the training and test datasets
print(f"Total training samples: {len(train_dataset)}")
print(f"Total test samples: {len(test_dataset)}")

Images batch shape: torch.Size([64, 3, 64, 64])
Labels batch shape: torch.Size([64])
Total training samples: 60000
Total test samples: 10000


In [36]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

In [37]:
optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=1e-3
)

criterion = nn.CrossEntropyLoss()
train_losses = []
test_losses = []

In [38]:
import time

images, labels = next(iter(train_loader))

optimizer.zero_grad()

t0 = time.perf_counter()
outputs = model(images)
t1 = time.perf_counter()

loss = criterion(outputs, labels)
t2 = time.perf_counter()

loss.backward()
t3 = time.perf_counter()

optimizer.step()
t4 = time.perf_counter()

print(f"Forward : {t1-t0:.3f}")
print(f"Backward: {t3-t2:.3f}")
print(f"Optimizer: {t4-t3:.3f}")

Forward : 0.164
Backward: 0.000
Optimizer: 0.001


In [39]:
epochs = 1

for epoch in tqdm(range(epochs), desc="Training Progress (epochs)"):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader):
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        # scheduler.step()
        running_loss += loss.item()

    train_losses.append(running_loss / len(train_loader))
    # get last learning rate from scheduler
    # print(scheduler.get_last_lr())
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

    test_losses.append(test_loss / len(test_loader))

    print(
        f"Epoch {epoch+1:2d} | "
        # f"LR={scheduler.get_last_lr()[0]:.6f} | "
        f"Train={train_losses[-1]:.4f} | "
        f"Val={test_losses[-1]:.4f}"
    )

Training Progress (epochs): 100%|██████████| 1/1 [03:43<00:00, 223.61s/it]

Epoch  1 | Train=0.8966 | Val=0.7033


In [40]:
model.eval()
correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:
        outputs = model(images)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7668
